# UC CosMx: annotation maps and protein check of the imputed 1K annotation

1. Attach the 1K-only and GCN-imputed (YWL-B v7) AUCell labels to the 1K objects; export the pseudo-1K WTX test set for AUCell
2. Spatial maps of the WTX, 6K + custom and 1K annotations (Fig 3B-C, Supp 6D, Supp 6G-H)
3. CD68 / PanCK protein vs RNA-based myeloid and epithelial labels (Spearman), and Otsu thresholds on the protein channels (Supp 6I)

In [ ]:
DATA_DIR = "/path/to/cosmx_data/uc_1k_6k_wtx"
K1_DIR   = DATA_DIR + "/1k"
WTX_DIR  = DATA_DIR + "/whole_trans"

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
from skimage.filters import threshold_otsu
import scipy.stats as stats

## Load objects and labels

In [ ]:
adata_wtx_1 = sc.read_h5ad(f'{WTX_DIR}/Processed_UC_P1_UC_P2/v7/h5ad_obj/qc_wtx_v7_UC_P1_UC_P2_anno.h5ad')
adata_1k_1 = sc.read_h5ad(f'{K1_DIR}/Processed_UC_P1_UC_P2/h5ad_img/imp_ywl_anno.h5ad')
adata_1k_2 = sc.read_h5ad(f'{K1_DIR}/Processed_UC_P3_UC_P4/h5ad_img/imp_ywl_anno.h5ad')
adata_1k = sc.read_h5ad(f'{K1_DIR}/Processed_merged/h5ad_obj/imp_ywl_anno.h5ad')
adata_6k_1_custom = sc.read_h5ad(f'{DATA_DIR}/6k/Processed_UC_P1_UC_P2_upd/h5ad_obj/qc_6k_UC_batch1_log1p_anno_6375genes.h5ad')

mapper = pd.read_csv('/path/to/scrna/uc/cell_type_name_mapper.csv')
ywl_1k_b = pd.read_csv(f'{K1_DIR}/Processed_merged/anno/ywl_imp_b_log_aucell.csv')
ywl_1k_b = ywl_1k_b.merge(mapper, left_on='label', right_on='cell_type_short', how='left')
anno_1k = pd.read_csv(f'{K1_DIR}/Processed_merged/anno/1kgenes_only_aucell.csv')[['cells', 'label']]
anno_1k = anno_1k.merge(mapper, left_on='label', right_on='cell_type_short', how='left')

## Attach the GCN-imputed (YWL-B) and 1K-only labels; write the annotated 1K objects

In [ ]:
IMP_COLS = {'v7_ywl_b_imp_cell_type_short': 'cell_type_short', 'v7_ywl_b_imp_cell_type_full': 'cell_type',
            'v7_ywl_b_imp_cell_category_from_type': 'cell_category', 'v7_ywl_b_imp_cell_type_general': 'cell_type_general'}
ONLY_1K_COLS = {'only_1k_cell_type': 'cell_type', 'only_1k_cell_cat_from_type': 'cell_category',
                'only_1k_cell_type_short': 'cell_type_short', 'only_1k_cell_type_general': 'cell_type_general'}
for col, src in IMP_COLS.items():
    adata_1k.obs[col] = ywl_1k_b[src].tolist()
for col, src in ONLY_1K_COLS.items():
    adata_1k.obs[col] = anno_1k[src].tolist()

# per-batch objects: same labels, matched by cell index (imputed) or by order (1K-only)
for ad, patients in [(adata_1k_1, ['UC_P1', 'UC_P2']), (adata_1k_2, ['UC_P3', 'UC_P4'])]:
    sub = adata_1k[adata_1k.obs['patient'].isin(patients)]
    ad.obs = ad.obs.drop(columns=list(IMP_COLS), errors='ignore').merge(
        sub.obs[list(IMP_COLS)], how='left', left_index=True, right_index=True)
    for col in ONLY_1K_COLS:
        ad.obs[col] = sub.obs[col].tolist()

adata_1k_1.write_h5ad(f'{K1_DIR}/Processed_UC_P1_UC_P2/h5ad_img/imp_ywl_anno.h5ad')
adata_1k_2.write_h5ad(f'{K1_DIR}/Processed_UC_P3_UC_P4/h5ad_img/imp_ywl_anno.h5ad')
adata_1k.write_h5ad(f'{K1_DIR}/Processed_merged/h5ad_obj/imp_ywl_anno.h5ad')

## Pseudo-1K WTX test set (1K genes only) for AUCell (`04_aucell_annotation.R`)

In [ ]:
test_wtx_1k = adata_wtx_1[adata_wtx_1.obs['train_test_split_pseudo_6k'] == 'Test']
test_wtx_1k = test_wtx_1k[:, test_wtx_1k.var_names.isin(adata_1k_1.var_names.tolist())].copy()
test_wtx_1k_df = pd.DataFrame(test_wtx_1k.X, columns=test_wtx_1k.var_names.tolist(),
                              index=test_wtx_1k.obs['fov_cell_id'].tolist())
test_wtx_1k_df.to_csv(f'{WTX_DIR}/Processed_UC_P1_UC_P2/v7/count_csv/psuedo_1k.csv')

## Sections shown: UC_P1 right tissue (WTX slide 4, 6K + custom slide 4, adjacent 1K section)

In [ ]:
# 6K + custom: FOV ranges of the four sections of batch 1
adata_6k_1_custom.obs['slide'] = [
    f"{i}_slide1" if int(j) <= 8 else f"{i}_slide2" if int(j) <= 34 else f"{i}_slide3" if int(j) <= 47 else f"{i}_slide4"
    for i, j in zip(adata_6k_1_custom.obs['orig.ident'], adata_6k_1_custom.obs['fov'])]

adata1_slide3_wtx = adata_wtx_1[adata_wtx_1.obs['slide'] == 'UC_batch1_slide4']
adata1_slide3_6k_custom = adata_6k_1_custom[adata_6k_1_custom.obs['slide'] == 'UC_batch1_slide4']
adata1_slide3_1k = adata_1k_1[adata_1k_1.obs['tissue'] == 'UC_P1_right']

## Spearman correlation of CD68 / PanCK protein with myeloid / epithelial labels (values printed on the maps)

In [ ]:
for name, ad, col in [('WTX', adata1_slide3_wtx, 'aucell_cell_category_from_type'),
                      ('1K no imputation', adata1_slide3_1k, 'only_1k_cell_cat_from_type'),
                      ('1K GCN imputation', adata1_slide3_1k, 'v7_ywl_b_imp_cell_category_from_type')]:
    for protein, category in [('Max.CD68', 'Myeloid'), ('Max.PanCK', 'Epithelial')]:
        rho, p = stats.spearmanr(ad.obs[protein], (ad.obs[col] == category).astype(int))
        print(f'{name}: {category} & {protein}: rho = {rho}, p = {p}')

## Plot helpers

In [ ]:
def category_map(ad, col, title='', legend_title='', corr_text=None, out=None, styled=True):
    """Cells coloured by a categorical column (tab20)."""
    coordinates = ad.obsm['spatial_fov']
    categories = ad.obs[col].astype('category')
    palette = sns.color_palette('tab20', len(categories.cat.categories))
    colors = categories.cat.codes.map(dict(enumerate(palette)))
    plt.figure(figsize=(8, 10))
    plt.scatter(coordinates[:, 0], coordinates[:, 1], c=colors, s=2)
    plt.title(title)
    plt.xlabel(''); plt.ylabel(''); plt.xticks([]); plt.yticks([]); plt.grid(False)
    handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette[i], markersize=10)
               for i in range(len(categories.cat.categories))]
    if styled:
        plt.legend(handles, categories.cat.categories, title=legend_title, loc='upper left', frameon=False,
                   fontsize=12, title_fontsize=14)
        for spine in plt.gca().spines.values():
            spine.set_visible(False)
    else:
        plt.legend(handles, categories.cat.categories, title=legend_title, loc='upper left', frameon=False)
    if corr_text:
        plt.text(0.98, 0.02, corr_text, transform=plt.gca().transAxes, ha='right', va='bottom', fontsize=12,
                 bbox=dict(facecolor='white', edgecolor='none', alpha=0.8, boxstyle='round,pad=0.3'))
    if out:
        plt.savefig(out, bbox_inches='tight', dpi=300)
    plt.show()


def protein_map(ad, col, title):
    """Cells coloured by a protein channel (viridis)."""
    coordinates = ad.obsm['spatial_fov']
    values = ad.obs[col]
    colors = plt.cm.viridis(plt.Normalize(vmin=values.min(), vmax=values.max())(values))
    plt.figure(figsize=(9, 10))
    scatter = plt.scatter(coordinates[:, 0], coordinates[:, 1], c=colors, s=2, cmap=plt.cm.viridis)
    plt.title(title)
    plt.xlabel(''); plt.ylabel(''); plt.xticks([]); plt.yticks([]); plt.grid(False)
    cbar = plt.colorbar(scatter, ax=plt.gca(), aspect=30, pad=0.02)
    cbar.set_label(col, rotation=270, labelpad=15)
    plt.show()

## Fig 3B: WTX annotation

In [ ]:
category_map(adata1_slide3_wtx, 'aucell_cell_category_from_type', legend_title='WTX',
             corr_text="Spearman's Correlation\nEpithelial cell & PanCK: 0.553 (p=0.0)\nMyeloid cell & CD68: 0.188 (p=5.0e-155)",
             out='WTX_slide4_annotation.jpg')

## Supp 6D: 6K + custom probes annotation, CD68 and PanCK

In [ ]:
category_map(adata1_slide3_6k_custom, 'aucell_cell_category_from_type', title='6K+200Custom UC_P1 Right Tissue Annotation',
             legend_title='Major Cell Type', styled=False)
protein_map(adata1_slide3_6k_custom, 'Max.CD68', '6K+200Custom UC_P2 Left Tissue CD68')
protein_map(adata1_slide3_6k_custom, 'Max.PanCK', '6K+200Custom UC_P2 Left Tissue PanCK')

## Supp 6G-H: 1K annotation without and with GCN imputation

In [ ]:
category_map(adata1_slide3_1k, 'only_1k_cell_cat_from_type', legend_title='1K No Imp',
             corr_text="Spearman's Correlation\nEpithelial cell & PanCK: 0.444 (p=0.0)\nMyeloid cell & CD68: 0.137 (p=4.3e-48)",
             out='1k_noimp_slide4_annotation.jpg')
category_map(adata1_slide3_1k, 'v7_ywl_b_imp_cell_category_from_type', legend_title='1K GCN Imp',
             corr_text="Spearman's Correlation\nEpithelial cell & PanCK: 0.499 (p=0.0)\nMyeloid cell & CD68: 0.229 (p=2.5e-133)",
             out='1k_ywl_imp_slide4_annotation.jpg')

## Fig 3C: FOV 44, fine cell types after imputation; CD68 and PanCK

In [ ]:
cell_types_order = [
    'B', 'ADAMDEC1+ Fib', 'Activated Fib', 'Crypt Top Fib', 'Cycl Fib',
    'Myofibroblast', 'OGN+RSPO3+ Fib', 'Pericyte', 'Senescent Fib',
    'Endothelial', 'Absoprtive Epi', 'Enteroendocrine', 'Inflamed Epi',
    'M cell', 'Secretive Epi', 'TA', 'Tuft',
    'Granulocyte', 'Cycl Myeloid', 'DC', 'Mo-mac', 'Neutrophil',
    'SELENOP+ Mac', 'NK', 'Neuron',
    'Plasma', 'CD4 Trm', 'CD8 Trm', 'DN T', 'Effector T',
    'Gd T', 'MAIT', 'Resting T', 'Tfh', 'Treg'
]
custom_palette_35 = [
    "#1f77b4", "#aec7e8", "#8fd1db", "#8ac0ed", '#ace8e3', "#a5daf2", "#9ad3d6", "#80b7b5", '#c6d6f5',
    "#ff7f0e", "#ffbb78", '#f5d2b5', "#f8c0a9", "#f1a3a3", "#e39b5f", "#e07b5b", "#df9f6e",
    "#2ca02c", "#98df8a", "#8fa72b", "#9abf62", "#a1d99b", "#6b9f67", "#d65a69", "#ff9b99",
    "#9467bd", "#ebc2ed", "#c0a4de", "#e377c2", "#9b6fa3", "#b175b6", "#d2a2b8", "#e4b5c9",
    "#e1ace8", "#b1b7d2"
]
category_column = 'v7_ywl_b_imp_cell_type_general'

adata_fov = adata_1k_1[adata_1k_1.obs["fov"].isin(["44"])].copy()
adata_fov.obs[category_column] = adata_fov.obs[category_column].replace('SELENOP+ Fib', 'Senescent Fib')
adata_fov = adata_fov[adata_fov.obs[category_column].isin(cell_types_order)]
adata_fov.obs[category_column] = pd.Categorical(adata_fov.obs[category_column], categories=cell_types_order, ordered=True)
adata_fov = adata_fov[adata_fov.obs[category_column].notna()]
adata_fov = adata_fov[adata_fov.obs.sort_values(by=category_column).index]
available_categories = adata_fov.obs[category_column].unique()
adata_fov.uns[f"{category_column}_colors"] = [custom_palette_35[cell_types_order.index(cat)] for cat in available_categories]

sq.pl.spatial_segment(adata_fov, color=category_column, library_key="fov", library_id=["44"],
                      seg_cell_id="cell_ID", seg_outline=True, ncols=3)
for protein in ["Max.PanCK", "Max.CD68"]:
    sq.pl.spatial_segment(adata_1k_1, color=protein, library_key="fov", library_id=["44"],
                          seg_cell_id="cell_ID", seg_outline=True, cmap=plt.cm.Reds)

## Supp 6I: CD68 and PanCK binarised with Otsu thresholds (1K section)

In [ ]:
for protein in ['Max.CD68', 'Max.PanCK']:
    vals = adata1_slide3_1k.obs[protein].values
    vals = vals[~np.isnan(vals)]
    otsu_thresh = threshold_otsu(vals)
    plt.figure(figsize=(8, 5))
    plt.hist(vals, bins=100, color='steelblue', alpha=0.8)
    plt.axvline(otsu_thresh, color='red', linestyle='--', label=f"Otsu Threshold = {otsu_thresh:.2f}")
    plt.xlabel(f"{protein.split('.')[1]} Expression")
    plt.ylabel('Number of Cells')
    plt.title(f"Histogram of {protein.split('.')[1]} Expression with Otsu Threshold")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# thresholds: Otsu values above (CD68 24524, PanCK 10460)
cd68, panck = adata1_slide3_1k.obs['Max.CD68'], adata1_slide3_1k.obs['Max.PanCK']
adata1_slide3_1k.obs['binary_prt_ct'] = np.where((cd68 >= 24524) & (panck < 10460), 'CD68+',
                                                 np.where((cd68 < 24524) & (panck >= 10460), 'PanCK+', 'Others'))

coordinates = adata1_slide3_1k.obsm['spatial_fov']
custom_palette = {'Others': 'lightgray', 'PanCK+': 'orange', 'CD68+': 'green'}
colors = adata1_slide3_1k.obs['binary_prt_ct'].astype('category').map(custom_palette)
plt.figure(figsize=(8, 10))
plt.scatter(coordinates[:, 0], coordinates[:, 1], c=colors, s=2)
plt.title("")
plt.xlabel(""); plt.ylabel(""); plt.xticks([]); plt.yticks([]); plt.grid(False)
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=custom_palette[label], markersize=10)
           for label in custom_palette]
plt.legend(handles, custom_palette.keys(), title="Protein (Otsu Threshold)", loc='upper left', frameon=False,
           fontsize=12, title_fontsize=14)
for spine in plt.gca().spines.values():
    spine.set_visible(False)
plt.savefig("1k_protein_slide4_annotation.jpg", bbox_inches='tight', dpi=300)
plt.show()